<center>
<h1>Chirundu Town Council CDF and Financial Dataset</h1>
<b>CSC 4792 Group Project - Group 39</b><br/>
University of Zambia<br/>
September 2026
</center>

---

## Overview

This consolidated notebook documents approved and proposed CDF projects, the 2025 decision list, annual budgets, financial statements, performance records and procurement plans. Different record types stay in separate tables so applications, allocations, estimates and actual expenditure are not confused.

**Sources:** Council documents in `data/source_inventory/download_inventory.json`; member notebooks archived under `data/interim/member_notebooks/`.

**Format:** UTF-8 CSV, pipe (`|`) separator and `db-unza26-csc4792-` filename prefix as required by the assignment.

**Reference notebook:** The numbered sections and pandas first-look/missing-value style follow *Starter Notebook: CS1 Failure Prediction Dataset* by Lighton Phiri (July 2026), referenced in the existing notebook. Council data and extraction methods are specific to this project.

---


## 1. Environment Setup

We import libraries for tables, PDF extraction and OCR. Extraction functions and manual corrections are included in this notebook. The procurement helper is also available as a script.

For a full run, keep this notebook with the project folders `data/raw/council_documents/`, `data/source_inventory/` and `models/`. Run it from the project folder or its `notebooks` subfolder, using the project's Python environment. Install the packages listed in `src/requirements-cdf-pilot.txt`, including `xlrd` for the 2026 workbook, if needed. Downloaded PDFs and the English OCR model are still required; this notebook is self-contained in code, not in source data.

Run the cells from top to bottom. OCR takes longer than loading a CSV. The financial and performance sections reuse saved OCR when available; their functions can recreate it from the PDFs.

In [1]:
# Import libraries
from pathlib import Path
from html.parser import HTMLParser
from urllib.parse import urljoin
import json
import re
import pandas as pd
import pdfplumber
import pypdfium2 as pdfium
from pypdf import PdfReader
from rapidocr_onnxruntime import RapidOCR
from IPython.display import display

pd.set_option('display.max_colwidth', 70)

# Update ROOT if the project is stored somewhere else
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

print('All libraries imported successfully.')
print('Project folder:', ROOT)

All libraries imported successfully.
Project folder: C:\Users\Arthur F Chipeta\Desktop\chirundu-cdf-project


## 2. Collect Document Links and Load the Inventory

The council webpages were saved as HTML before extraction. We read their links, keep document downloads, combine duplicate URLs and record each source page. The parser below was previously in a separate script and is now part of this notebook.

The observed links were reviewed and organised into `download_inventory.json`, with a stable inventory number, category and filename for each selected source. This was a manual source-selection step. Files were downloaded using the supplied `data/source_inventory/download_documents.ps1`, which skips existing files. That collection script remains part of the codebase; it is not needed to run extraction once the files are present.

Re-running this section reads the saved pages rather than changing the collection to whatever is currently online. The URLs and page snapshots document where the files came from. The notebook does not require any local Python module.

The reference notebook starts by loading a finished CSV. Here we first create the CSVs from the source PDFs; section 8 loads the saved results for inspection.

In [2]:
class Links(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self.href = None
        self.parts = []
    def handle_starttag(self, tag, attrs):
        if tag == 'a':
            self.href = dict(attrs).get('href')
            self.parts = []
    def handle_data(self, data):
        if self.href is not None:
            self.parts.append(data)
    def handle_endtag(self, tag):
        if tag == 'a' and self.href is not None:
            self.links.append((' '.join(' '.join(self.parts).split()), self.href))
            self.href = None

In [3]:
# Read links from the saved official webpages
var_source_folder = ROOT / 'data/source_inventory'
var_base_url = 'https://www.chirunducouncil.gov.zm/'
var_observed = {}

for var_page_path in sorted(var_source_folder.glob('*.html')):
    var_page_id = '959' if var_page_path.stem == 'publications' else var_page_path.stem.removeprefix('page-')
    var_source_url = var_base_url + '?page_id=' + var_page_id
    var_parser = Links()
    var_parser.feed(var_page_path.read_text(encoding='utf-8-sig'))

    for var_label, var_href in var_parser.links:
        var_url = urljoin(var_source_url, var_href)
        if '/wp-content/uploads/' not in var_url:
            continue
        if not var_url.lower().split('?')[0].endswith(('.pdf', '.xls', '.xlsx', '.doc', '.docx', '.csv')):
            continue
        var_key = var_url.removeprefix('http://').removeprefix('https://')
        if var_key not in var_observed:
            var_observed[var_key] = {'url': var_url, 'labels': [], 'source_pages': []}
        if var_label and var_label not in var_observed[var_key]['labels']:
            var_observed[var_key]['labels'].append(var_label)
        if var_source_url not in var_observed[var_key]['source_pages']:
            var_observed[var_key]['source_pages'].append(var_source_url)

var_links_df = pd.DataFrame(var_observed.values())
print('Unique document links found:', len(var_links_df))
display(var_links_df.head())

Unique document links found: 104


,url,labels,source_pages
0,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2022-...,[2022-Approved-Community-Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]
1,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2023-...,[2023-Approved-Community-Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]
2,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2024-...,[2024-Approved-Community-Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]
3,http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/06/2025-...,[2025-Approved-Community-Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]
4,http://www.chirunducouncil.gov.zm/wp-content/uploads/2026/05/2026-...,[2026 Approved Community Projects],[https://www.chirunducouncil.gov.zm/?page_id=3241]


In [4]:
var_inventory_file = ROOT / 'data/source_inventory/download_inventory.json'
var_inventory = json.loads(var_inventory_file.read_text(encoding='utf-8'))
var_inventory_df = pd.DataFrame(var_inventory)
var_selected_ids = [1, 2, 3, 59, 5, 104, 34, 33, 32, 30, 39, 38, 35, 31, 36, 75]
display(var_inventory_df.loc[var_inventory_df['inventory_id'].isin(var_selected_ids),
                             ['inventory_id', 'display_title', 'url']])

,inventory_id,display_title,url
0,1,2022-Approved-Community-Projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2022-...
1,2,2023-Approved-Community-Projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2023-...
2,3,2024-Approved-Community-Projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/2024-...
4,5,2026 Approved Community Projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2026/05/2026-...
9,59,2025 CDF approved projects,http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/11/2025-...
16,30,2026 OBB Budget K112.4Mn,http://www.chirunducouncil.gov.zm/wp-content/uploads/2026/02/2026-...
17,31,Bi Annual Performance Report,http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/12/2025-...
18,32,Chirundu 2025 OBB Annual Budget Final - (K103.9 million),http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/11/Chiru...
19,33,Chirundu Town Council OBB Budget 2024 (Final),http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/08/Chiru...
20,34,Chirundu Town Council OBB Budget 2023 (Final),http://www.chirunducouncil.gov.zm/wp-content/uploads/2024/11/Chiru...


## 3. Extract and Clean the Approved CDF Project Lists

We begin with the detailed 2025 list, then repeat the method for 2022, 2023, 2024 and 2026. These are entries in annual approved lists, not necessarily different physical projects.

### 3.1 Read the 2025 page

The PDF is scanned, so we render its page as an image. OCR returns text and its positions. The positions let us put the text back into table rows and columns.

In [5]:
source_url = 'http://www.chirunducouncil.gov.zm/wp-content/uploads/2025/11/2025-Approved-Community-Projects-.pdf'

pdf_path = ROOT / 'data/raw/council_documents/01_cdf_community_projects/059--2025-Approved-Community-Projects-.pdf'
folder = ROOT / 'data/interim/ocr_2025'
folder.mkdir(parents=True, exist_ok=True)
pdf = pdfium.PdfDocument(pdf_path)
page_image = pdf[0].render(scale=2).to_pil()
image_path = folder / 'page_1.png'
page_image.save(image_path)
pdf.close()

print('PDF text:', PdfReader(pdf_path).pages[0].extract_text())
print('Image size:', page_image.size)

PDF text: CamScanner

Image size: (1684, 1190)


In [6]:
engine = RapidOCR(rec_model_path=str(ROOT / 'models/en_PP-OCRv3_rec_infer.onnx'), intra_op_num_threads=2, inter_op_num_threads=2, det_limit_side_len=1684)
result, elapsed = engine(str(image_path))
if not result:
    raise ValueError('OCR did not find any text on the page.')
(folder / 'ocr_text.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')

print('Text sections found:', len(result))
for box, text, score in result[:10]:
    print(text)

Text sections found: 114
2025 CDF Approved Community Projects
S.No
Project Name
Ward
Zone
Location
Comment
Construction of Solar powered
Ibbwemunyama Rural
1


### 3.2 Put OCR text into rows

The boundaries below were measured from the page rendered at scale 2. They describe the table layout, not the project values. A small position adjustment follows the slope of the scanned table.

In [7]:
# Pixel boundaries for the six columns and fourteen rows on this page.
columns = [250, 572, 738, 920, 1128, 1466]
row_edges = [154, 211, 293, 321, 377, 463, 545, 574, 659, 743, 773, 860, 918, 947, 1028]
names = ['project_name', 'ward', 'zone', 'location', 'approval_comment']
rows = []
for index, (top, bottom) in enumerate(zip(row_edges[:-1], row_edges[1:]), 1):
    row = {'project_no': index}
    for column, left, right in zip(names, columns[:-1], columns[1:]):
        words = []
        for box, text, confidence in result:
            x = sum(point[0] for point in box) / 4
            y = sum(point[1] for point in box) / 4
            # The scanned grid slopes slightly upwards towards the right.
            y += 14 * max(0, y - 150) / 878 * max(0, x - 250) / 1216
            if left <= x < right and top <= y < bottom:
                words.append((y, x, text))
        row[column] = ' '.join(word[2] for word in sorted(words)) or None
    rows.append(row)
df = pd.DataFrame(rows)
df.to_csv(folder / 'ocr_table_raw.csv', sep='|', index=False)
raw_df = df.copy()
display(raw_df.head())

,project_no,project_name,ward,zone,location,approval_comment
0,1,Construction of Solar powered borehole,Ibbwemunyama,Ibbwemunyama,Ibbwemunyama Rural Health Post,Approved
1,2,Community water project,Chirundu west,Chibende and Lusumpuko,Chibende and Lusumpuko Communities,Approved
2,3,Completion of Maternity Wing,Chirundu West,Chibende,Chibende,Approved
3,4,Construction of a 1x3 classroom block,Kapululira,Farao,Farao Community School,Approved
4,5,Construction of water borne toilets and solar powered borehole,Njame,Chibulameenda,NaN,Approved


### 3.3 Clean the 2025 records

We trim extra spaces, remove duplicates and apply the corrections read from the source image. OCR joined some cells and included stamp text. These manual corrections remain visible so the cleaning can be explained.

The approval comment for project 11 permits only the motor grader. No project amounts or physical implementation statuses are stated in this list, so they remain missing.

In [8]:
df = raw_df.copy()
for column in names:
    df[column] = df[column].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()

# Corrections read from the source page.
corrections = {
    (6, 'project_name'): 'Construction of a Double Storey Classroom Block (3 classes per storey with Offices)',
    (6, 'ward'): 'Ngombe Illede',
    (6, 'location'): 'Proposed Ngombe Illede Sec. School',
    (9, 'location'): '10 – Sikoongo zone, 04-Siachibubba zone and 06-T-Junction zone',
    (11, 'approval_comment'): 'Approved procurement of Motor grader',
    (12, 'approval_comment'): 'Approved',
    (13, 'approval_comment'): 'Approved',
    (14, 'approval_comment'): 'Approved',
}
for (project_no, column), value in corrections.items():
    df.loc[df['project_no'] == project_no, column] = value

df['ward'] = df['ward'].str.title()
df = df.drop_duplicates(subset=names).copy()
display(df.head())

,project_no,project_name,ward,zone,location,approval_comment
0,1,Construction of Solar powered borehole,Ibbwemunyama,Ibbwemunyama,Ibbwemunyama Rural Health Post,Approved
1,2,Community water project,Chirundu West,Chibende and Lusumpuko,Chibende and Lusumpuko Communities,Approved
2,3,Completion of Maternity Wing,Chirundu West,Chibende,Chibende,Approved
3,4,Construction of a 1x3 classroom block,Kapululira,Farao,Farao Community School,Approved
4,5,Construction of water borne toilets and solar powered borehole,Njame,Chibulameenda,<NA>,Approved


In [9]:
df['year'] = 2025
df['amount_zmw'] = pd.NA
df['implementation_status'] = pd.NA
df['source_url'] = source_url
df['source_page'] = 1
df = df[['project_no', 'year', 'project_name', 'ward', 'zone', 'location',
         'approval_comment', 'amount_zmw', 'implementation_status', 'source_url', 'source_page']]


print('Cleaned 2025 records:', len(df))

Cleaned 2025 records: 14


### 3.4 Set the layouts for the other years

The tables change between PDFs. `CDF_LAYOUTS` gives their column boundaries and numbered rows. The second 2023 page needs a 180-degree rotation, and the 2026 ward headings are excluded. These measurements must change if a different document is used.

In [10]:
def numbered(edges, first=1):
    return [(first+i, a, b) for i, (a, b) in enumerate(zip(edges[:-1], edges[1:]))]

CDF_LAYOUTS = [
    dict(year=2022, source_id=1, page=1, rotate=0,
         columns={'project_name':(105,389),'project_description':(389,805),'sector':(805,904),'ward':(1084,1183),'zone':(1183,1333),'location':(1333,1596)},
         rows=numbered([278,348,400,435,469,509,559,596,628,681,739,806,858,945,986,1056,1093]), tilt=(7,-0.01,278)),
    dict(year=2023, source_id=2, page=1, rotate=0,
         columns={'project_name':(104,523),'project_description':(523,759),'sector':(759,964),'ward':(1082,1202),'zone':(1202,1350),'location':(1350,1625)},
         rows=numbered([179,209,256,317,365,395,426,506,597,644,704,765,826,887,947,1023,1131]), tilt=(14,-0.016,179)),
    dict(year=2023, source_id=2, page=2, rotate=180,
         columns={'project_name':(120,534),'project_description':(534,770),'sector':(770,973),'ward':(1090,1211),'zone':(1211,1360),'location':(1360,1644)},
         rows=numbered([217,404,510,619],17), tilt=(-8,0,217)),
    dict(year=2024, source_id=3, page=1, rotate=0,
         columns={'project_name':(160,855),'sector':(855,1113),'ward':(1503,1618)},
         rows=numbered([188,230,279,311,344,391,455,519,600,713,825,867,926,944,961,978]), tilt=(8,-0.034,188)),
    dict(year=2026, source_id=5, page=1, rotate=0,
         columns={'project_name':(258,502),'ward':(502,734),'zone':(734,874),'location':(874,1059),'work_item':(1059,1518)},
         rows=[(1,200,360),(2,395,529),(3,559,641),(4,670,751),(5,782,866),(6,895,976)], tilt=(0,0,0)),
    dict(year=2026, source_id=5, page=2, rotate=0,
         columns={'project_name':(261,502),'ward':(502,731),'zone':(731,870),'location':(870,1057),'work_item':(1057,1523)},
         rows=[(7,96,170),(8,198,252),(9,281,338),(10,368,449),(11,482,599),(12,645,730),(13,734,788),(14,821,935),(15,937,1048)], tilt=(-18,0,0)),
    dict(year=2026, source_id=5, page=3, rotate=0,
         columns={'project_name':(197,475),'ward':(475,738),'zone':(738,891),'location':(891,1105),'work_item':(1105,1608)},
         rows=[(16,130,224),(17,225,287),(18,319,378),(19,409,443)], tilt=(-23,0,0)),
]

### 3.5 Extract the remaining lists

The first function groups words using a row's measured boundaries. The second renders each selected page, runs OCR and collects its records. We locate PDFs by their inventory prefix so the code does not depend on the original computer's absolute file paths.

In [11]:
def group_page(result, layout, source_url):
    rows = []
    for number, top, bottom in layout['rows']:
        row = {'project_no':number, 'year':layout['year'], 'source_url':source_url, 'source_page':layout['page']}
        for name, (left,right) in layout['columns'].items():
            words = []
            for box,text,score in result:
                x = sum(p[0] for p in box)/4
                y = sum(p[1] for p in box)/4
                offset, slope, origin = layout['tilt']
                y -= (offset+slope*(y-origin))*max(0,x-100)/1500
                if left <= x < right and top <= y < bottom:
                    # Bucket nearly level text together, then read left to right.
                    words.append((round(y/10),x,text))
            row[name] = ' '.join(w[2] for w in sorted(words)) or None
        rows.append(row)
    return rows

In [12]:
def extract_remaining_years():
    inventory = json.loads((ROOT/'data/source_inventory/download_inventory.json').read_text(encoding='utf-8'))
    sources = {r['inventory_id']: r for r in inventory}
    folder = ROOT/'data/interim/cdf_other_years'
    folder.mkdir(parents=True, exist_ok=True)
    engine = RapidOCR(rec_model_path=str(ROOT/'models/en_PP-OCRv3_rec_infer.onnx'),
                      intra_op_num_threads=2, inter_op_num_threads=2, det_limit_side_len=1684)
    rows = []
    for layout in CDF_LAYOUTS:
        source = sources[layout['source_id']]
        pdf_path = next((ROOT/'data/raw/council_documents').rglob(f"{layout['source_id']:03d}--*"))
        pdf = pdfium.PdfDocument(pdf_path)
        image = pdf[layout['page']-1].render(scale=2).to_pil()
        if layout['rotate']:
            image = image.rotate(layout['rotate'], expand=True)
        image_path = folder/f"{layout['year']}_page_{layout['page']}.png"
        image.save(image_path)
        pdf.close()
        result, _ = engine(str(image_path))
        if not result:
            raise ValueError(f'No text found: {image_path.name}')
        image_path.with_suffix('.json').write_text(json.dumps(result, ensure_ascii=False, indent=2),encoding='utf-8')
        rows.extend(group_page(result, layout, source['url']))
        print(f"Read {layout['year']} page {layout['page']}: {len(layout['rows'])} table rows", flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(folder/'raw_projects.csv',sep='|',index=False)
    return df

In [13]:
raw_other_years = extract_remaining_years()
display(raw_other_years.groupby('year').size().rename('records'))

Read 2022 page 1: 16 table rows


Read 2023 page 1: 16 table rows


Read 2023 page 2: 3 table rows


Read 2024 page 1: 15 table rows


Read 2026 page 1: 6 table rows


Read 2026 page 2: 9 table rows


Read 2026 page 3: 4 table rows


year
2022    16
2023    19
2024    15
2026    19
Name: records, dtype: int64

### 3.6 Correct OCR errors and keep missing values

The following edits were read from the page images. A year and printed project number identify each correction. Text that is cut off or hidden by a stamp is noted rather than invented. Spelling differences in the source remain unless the difference is an OCR error.

In [14]:
CORRECTIONS = {}

In [15]:
# Corrections for 2022
CORRECTIONS[2022] = {5: {'project_name': 'DRILLING AND EQUIPPING OF SOLAR POWERED BOREHOLE AT HACHIBBUBA'},
 6: {'project_name': 'DRILLING AND EQUIPPING OF SOLAR POWERED BOREHOLE AT T-JUNCTION'},
 9: {'project_name': 'CONSTRUCTION OF 1X3 CLASSROOM BLOCK AT HAMBUTO PRIMARY SCHOOL',
     'project_description': 'Construction of 1x3 CRB at Hambuto Primary School',
     'ward': 'IBBWEMUNYAMA'},
 10: {'project_name': 'CONSTRUCTION OF STAFF HOUSE, ABLUTION BLOCK AND DRILLING AND EQUIPPING OF '
                      'SOLAR POWERED BOREHOLE AT VELU HEALTH POST'},
 11: {'zone': 'MANDENGA'},
 12: {'project_description': 'Construction of Semi-detached Staff House at Machavika Primary '
                             'School'},
 13: {'project_name': 'CONSTRUCTION OF 1X3 CLASSROOM BLOCK, ABLUTION BLOCK, STAFF HOUSE, DRILLING '
                      'AND EQUIPPING OF SOLAR POWERED BOREHOLE AT KATWEZELE PRIMARY SCHOOL',
      'project_description': 'Construction of 1no 1x3 CRB, 1no Staff House, 1no Ablution block and '
                             'drilling/equipping of 1no Solar powered borehole at Katwezele '
                             'Community School',
      'location': None},
 14: {'location': None},
 15: {'project_description': 'SUPPLY AND DELIVERY OF DESKS WORTH ZMW1,746,363.23',
      'location': None},
 16: {'project_name': 'CONSTRUCTION OF 1X2 CRB AT KATWEZELE PRIMARY SCHOOL',
      'location': 'KATWEZELE COMMUNITY SCHOOL'}}

In [16]:
# Corrections for 2023
CORRECTIONS[2023] = {7: {'project_description': 'CONSTRUCTION OF ABLUTION BLOCK WITH SOLAR POWERED WATER SUPPLY AT '
                            'T-JUNCTION',
     'sector': 'SANITATION',
     'ward': 'CHIRUNDU CENTRAL',
     'zone': 'MANDENGA'},
 8: {'project_name': 'CONSTRUCTION 2 OF ABLUTION BLOCKS AT MISSION MARKET AND YELLOW [text cut '
                     'off]'},
 10: {'project_description': 'PROCUREMENT OF 150 DESKS FOR ZALAUNGA PRIMARY SCHOOL'},
 11: {'project_description': 'PROCUREMENT OF 200 DESKS FOR NYANZALA PRIMARY SCHOOL'},
 13: {'project_description': 'PROCUREMENT OF 200 DESKS AT MAUNGA PRIMARY SCHOOL',
      'ward': 'IBBWEMUNYAMA',
      'zone': 'MAUNGA',
      'location': 'MAUNGA COMMUNITY SCHOOL'},
 14: {'project_name': 'PROCUREMENT OF DESKS AT 4 MILES COMMUNITY SCHOOL',
      'project_description': 'PROCUREMENT OF 100 DESKS FOR FOUR MILES COMMUNITY SCHOOL',
      'ward': 'CHIRUNDU WEST',
      'zone': 'LUSUMPUKO',
      'location': 'FOUR MILES SCHOOL'},
 15: {'ward': 'CHIRUNDU WEST', 'zone': 'CHIBENDE', 'location': 'CHIBENDE CLINIC'},
 16: {'zone': 'PAMBAZANA'},
 17: {'project_description': 'Rehabilitation of Lusitu Water System to include Chilindi, '
                             'Machavika, Chibulameenda, extension from Siamaundu to Siabulembo',
      'ward': 'LUSITU, NJAME, NGOMBE ILEDE'},
 18: {'project_description': 'DRILLING AND EQUIPPING OF SOLAR POWRED BOREHOLE AT NABBANDA CLINIC',
      'sector': 'WATER'},
 19: {'project_name': 'INSTALLATION OF WATER SUPPLY NETWORK IN CHIRUNDU CENTRAL WARD',
      'project_description': 'EXTENSION OF WATER SUPPLY IN KANENGUMBO, KADUNGA AND MANDENGA '
                             'VILLAGE',
      'sector': 'WATER'}}

In [17]:
# Corrections for 2024
CORRECTIONS[2024] = {1: {'project_name': 'CONSTRUCTION OF 1X2 CRB, INSTALLATION AND EQUIPPING OF SOLAR POWERED '
                     'BOREHOLE, CONSTRUCTION OF TOILETS, AT ZALAUNGA PRIMARY SCHOOL'},
 2: {'project_name': 'CONSTRUCTION OF 1X3 CRB AT SIKOONGO SKILLS CENTER'},
 3: {'ward': 'SIKOONGO'},
 5: {'project_name': 'CONSTRUCTION OF 1X3 CRB & ABLUTION BLOCK AT NAMABUYU SCHOOL',
     'ward': 'NJAME'},
 11: {'project_name': 'INSTALLATION OF SOLAR POWERED WATER SYSTEM IN KAPULULIRA'},
 12: {'project_name': 'CONSTRUCTION OF 1X3 CRB, ABLUTION BLOCK, INSTALLATION AND EQUIPPING OF '
                      'SOLAR POWERED BOREHOLES AT CHIPEPO COMMUNITY SCHOOL'},
 15: {'project_name': 'PROCUREMENT OF MOTORBIKES'}}

In [18]:
# Corrections for 2026
CORRECTIONS[2026] = {1: {'project_name': 'Construction of 1x3 CRB, 1 Staff house, A solar powered borehole and '
                     'Maternity Annex'},
 2: {'zone': 'Chilindi; Namabuyu',
     'location': 'Chilindi Primary School; Namabuyu Primary School',
     'work_item': 'Staff houses; 1 solar powered borehole'},
 3: {'project_name': 'Construction of sports facility (phase 2)'},
 5: {'work_item': 'Siabulembo road'},
 6: {'work_item': '1x3 CRB'},
 16: {'project_name': 'Variation (Construction of 2 courses and columns at stadium)'},
 19: {'project_name': 'Procurement of Desks'}}

In [19]:
NOTES = {
    (2022,13): 'Location text is obscured by the council stamp; left blank.',
    (2022,14): 'Stamp text was removed from the location cell; no location recovered.',
    (2022,15): 'Amount is the reported value of desk supply, not confirmed expenditure. Stamp text was removed from the location cell.',
    (2022,16): 'Project name says Primary School; location says Community School. Source wording retained.',
    (2023,8): 'Project-name text is cut off at the column edge; the description identifies Mission and Yellow Jacket markets.',
    (2023,17): 'One source entry covers multiple wards; it has not been split into three projects.',
    (2023,18): 'The location cell ends CLINI in the source. Source spelling retained.',
    (2023,19): 'The project-name ending was read with the ward column. Description says KANENGUMBO; location says KANEGUMBO. Source spellings retained.',
    (2026,2): 'One numbered entry covers two schools; sites and corresponding work items are separated by semicolons in source order.',
    (2026,8): 'Project/zone spell Shangwemu; location spells Shyangwemu. Source wording retained.',
    (2026,10): 'Project name says Mateaunga; location says Mateaungu. Source wording retained.',
    (2026,12): 'Project name spells Hachibubba; zone/location spell Hachibbuba. Source wording retained.',
    (2026,14): '2026 list entry refers to completion work on a 2024 project; it is not evidence that the work is complete.',
    (2026,15): '2026 list entry refers to completion work on a 2024 project; it is not evidence that the work is complete.',
}

In [20]:
def clean_remaining_years(raw):
    df = raw.copy()
    for year, projects in CORRECTIONS.items():
        for number, changes in projects.items():
            for column, value in changes.items():
                df.loc[(df.year == year) & (df.project_no == number), column] = value
    text_columns = ['project_name','project_description','sector','ward','zone','location','work_item']
    for column in text_columns:
        df[column] = df[column].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    # Repair inconsistent spacing in the same literal all-wards label.
    for column in ['ward','zone','location']:
        df[column] = df[column].str.replace(r'(?i)ALL\s*12\s*WARDS','All 12 Wards',regex=True)
    df['ward'] = df['ward'].str.replace(r'(?i)CHIRUNDU\s*-\s*WEST','Chirundu West',regex=True).str.title()
    df['sector'] = df['sector'].str.title()
    # The wording 'worth ZMW...' gives a supply value, not an allocation or payment.
    values = df['project_description'].str.extract(r'(?i)worth\s+ZMW\s*([\d,]+\.\d{2})',expand=False)
    df['amount_zmw'] = pd.to_numeric(values.str.replace(',','',regex=False), errors='coerce')
    df['amount_type'] = pd.NA
    df.loc[df.amount_zmw.notna(),'amount_type'] = 'Reported supply value'
    df['implementation_status'] = pd.NA
    df['approval_comment'] = pd.NA
    df['approval_status'] = 'Approved list'
    df['notes'] = [NOTES.get((r.year,r.project_no),'') for r in df.itertuples()]
    return df.drop_duplicates(subset=['year','project_no','source_url','source_page'])

In [21]:
other_years = clean_remaining_years(raw_other_years)
display(other_years.groupby('year').size().rename('records'))

year
2022    16
2023    19
2024    15
2026    19
Name: records, dtype: int64

### 3.7 Combine the annual lists and export

We combine the five annual lists with `pd.concat()`, sort by year and project number, and remove duplicate source rows. A project name containing “completion” is not evidence of completed work.

The one reported monetary amount is a desk-supply value in the 2022 list. It must not be interpreted as confirmed expenditure. The earlier pilot sample is not added as another dataset.

In [22]:
projects_2025 = df.copy()
projects_2025['approval_status'] = 'Approved list'
projects_2025.loc[projects_2025['project_no'] == 11, 'approval_status'] = 'Approved with scope restriction'
projects_2025['notes'] = ''
projects_2025.loc[projects_2025['project_no'] == 11, 'notes'] = 'Only procurement of the motor grader is approved in the source comment.'

combined = pd.concat([other_years, projects_2025], ignore_index=True)
columns = ['project_no', 'year', 'project_name', 'project_description', 'sector',
           'ward', 'zone', 'location', 'work_item', 'approval_status', 'approval_comment',
           'amount_zmw', 'amount_type', 'implementation_status', 'source_url', 'source_page', 'notes']
combined = combined[columns].sort_values(['year', 'project_no']).reset_index(drop=True)
combined = combined.drop_duplicates(subset=['year', 'project_no', 'source_url', 'source_page'])

print('Records by year:')
print(combined.groupby('year').size().to_string())
print('Total records:', len(combined))
print('Missing project names:', combined['project_name'].isna().sum())
print('Missing ward values:', combined['ward'].isna().sum())

output_folder = ROOT / 'data/processed/cdf_projects'
output_folder.mkdir(parents=True, exist_ok=True)
output_file = output_folder / 'db-unza26-csc4792-chirundu_cdf_approved_projects_2022_2026.csv'
try:
    combined.to_csv(output_file, sep='|', index=False, encoding='utf-8')
except PermissionError:
    # A desktop preview can hold this already-generated CSV open on Windows.
    from io import StringIO
    saved = pd.read_csv(output_file, sep='|')
    expected = pd.read_csv(StringIO(combined.to_csv(sep='|', index=False)), sep='|')
    pd.testing.assert_frame_equal(saved, expected, check_dtype=False)
    print('Existing approved-project CSV is current; Windows has it open.')
print('Saved:', output_file.name)

Records by year:
year
2022    16
2023    19
2024    15
2025    14
2026    19
Total records: 83
Missing project names: 0
Missing ward values: 3
Existing approved-project CSV is current; Windows has it open.
Saved: db-unza26-csc4792-chirundu_cdf_approved_projects_2022_2026.csv
